# 01 — Data Cleaning & Validation
Load the source dataset, inspect every column, fix quality issues, validate, export v3.

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/clean_brick_dataset_v2.csv')
df.shape


(361, 14)

In [4]:
for col in df.columns:
    print(f"===== {col} =====")
    print(df[col].value_counts(dropna=False).head(15))
    print()
    

===== Districts =====
Districts
Medchal Malkajgiri        43
Hyderabad                 33
Yadadri Bhuvanagiri       27
Hanumakonda               22
Peddapalli                21
Karimnagar                20
Mancherial                16
Jangaon                   14
Nalgonda                  13
Kumuram Bheem Asifabad    11
Sangareddy                11
Khammam                   11
Jagtial                   10
Siddipet                  10
Bhadradri Kothagudem      10
Name: count, dtype: int64

===== Location =====
Location
Nerpalle, Telangana 504292, India                                                                                                                                 2
Gattusingaram, Telangana 505188, India                                                                                                                            2
10-91/c, Bommakal Rd, Bommakal, Karimnagar, Telangana 505002, India                                                                                 

In [5]:
dupes = df[df.duplicated(subset=['Company Name', 'Location'], keep=False)]
print(f"Rows involved in potential duplicates: {len(dupes)}")
dupes.sort_values('Company Name')[['Company Name', 'Location', 'Districts', 'Status', 'Contact']]

Rows involved in potential duplicates: 12


,Company Name,Location,Districts,Status,Contact
261,GRK Manufacturers pvt ltd,"Girmapur, Medchal, Secunderabad, Telangana 501...",Medchal Malkajgiri,Running,9666811162
303,GRK Manufacturers pvt ltd,"Girmapur, Medchal, Secunderabad, Telangana 501...",Hyderabad,Running,9666811162
258,Interlocking Cement bricks,"SVNO: 52, a/2, Kundanpally, Keesara, Secundera...",Medchal Malkajgiri,Running,9000800248
316,Interlocking Cement bricks,"SVNO: 52, a/2, Kundanpally, Keesara, Secundera...",Hyderabad,Running,9000800248
234,Laxmi Narasimha Cement Bricks Manufacturer,"Unnamed Road, Laxmapur, Telangana 508117, India",Yadadri Bhuvanagiri,Running,9963608839
270,Laxmi Narasimha Cement Bricks Manufacturer,"Unnamed Road, Laxmapur, Telangana 508117, India",Medchal Malkajgiri,Running,9963608839
341,Rangam Narasimha Bricks company,"Gopalpet Rd, Wanaparthy, Telangana 509106, India",Narayanpet,Running,9948535182
354,Rangam Narasimha Bricks company,"Gopalpet Rd, Wanaparthy, Telangana 509106, India",Wanaparthy,Running,9948535182
257,Shree Vazra Enterprises(AAC brick manufacturer),"Shamirpet - Keesara Rd, Secunderabad, Telangan...",Medchal Malkajgiri,Running,6305034270
315,Shree Vazra Enterprises(AAC brick manufacturer),"Shamirpet - Keesara Rd, Secunderabad, Telangan...",Hyderabad,Running,6305034270


In [6]:
before = len(df)
df = df.drop_duplicates(subset=['Company Name', 'Location'], keep='first').reset_index(drop=True)
after = len(df)
print(f"Dropped {before - after} duplicate rows: {before} → {after}")

Dropped 6 duplicate rows: 361 → 355


**Decision — duplicates:** 6 factory pairs appeared twice with identical name/address/phone but conflicting districts (border-area scrape overlap). Kept first occurrence; district for these 6 should be treated as approximate.

In [7]:
# Strip whitespace from every text column (kills bug #2 everywhere, found or not)
for col in df.select_dtypes(include=['object', 'str']).columns:
    df[col] = df[col].str.strip()

# Fix casing on the three diagnosed columns
df['Company Type'] = df['Company Type'].str.title()
df['Company Category'] = df['Company Category'].str.replace('fly Ash', 'Fly Ash')
df['Status'] = df['Status'].str.title()


In [8]:
for col in ['Company Type', 'Machine Type', 'Company Category', 'Status']:
    print(f"===== {col} =====")
    print(df[col].value_counts(dropna=False))
    print()

===== Company Type =====
Company Type
Manufacturing    354
Not Specified      1
Name: count, dtype: int64

===== Machine Type =====
Machine Type
Hand Made                               198
Semi Automatic Concrete Machine Made     76
Fully Automatic Machine Made             64
Not Specified                            17
Name: count, dtype: int64

===== Company Category =====
Company Category
Fly Ash Brick Plant                    169
Traditional Clay Factory                99
Fully Automatic Fly Ash Brick Plant     73
Not Specified                           14
Name: count, dtype: int64

===== Status =====
Status
Running               303
Permanently Closed     26
Temporarily Closed     26
Name: count, dtype: int64



In [9]:
df = df.replace('Not Specified', np.nan)

missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})

,missing_count,missing_pct
Founder,339,95.5
Website,308,86.8
Contact,150,42.3
Product Type,18,5.1
Block Type,18,5.1
Machine Type,17,4.8
Company Size,17,4.8
Company Category,14,3.9
Location,1,0.3
Company Type,1,0.3


In [10]:
df['has_website'] = df['Website'].notna()
df['has_contact'] = df['Contact'].notna()

df = df.drop(columns=['Founder'])

for col in ['Machine Type', 'Product Type', 'Block Type', 'Company Category', 'Company Size']:
    df[col] = df[col].fillna('Unknown')

df.shape

(355, 15)

In [11]:
assert len(df) == 355, "Row count wrong"
assert df['Status'].nunique() == 3, "Status has wrong number of values"
assert not df['Districts'].isna().any(), "Districts has nulls"
assert not df['Company Type'].str.contains('manufacturing', case=True, na=False).any(), "Casing bug remains"
assert df['Machine Type'].str.startswith(' ').sum() == 0, "Whitespace bug remains"
print("All assertions passed ✓")

All assertions passed ✓
